<a href="https://colab.research.google.com/github/Narendra725/Power_BI_Spark_Labs/blob/main/Power%20BI/Automations/Power%20Bi%20Desktop/Mach3/power_bi_objects_creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [116]:
# Centralized Path Configuration
import os

# Your Git Repo Details
REPO_URL = "https://github.com/Narendra725"
REPO_NAME = "Power_BI_Spark_Labs"

# Setting the working root to Mach3
REPO_DIR = "/content/Power_BI_Spark_Labs"
MACH3_ROOT = os.path.join(REPO_DIR, 'Power BI/Automations/Power Bi Desktop/Mach3')

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}/{REPO_NAME}.git
    print(f"Successfully cloned {REPO_NAME}")

# Create Mach3 if it doesn't exist
os.makedirs(MACH3_ROOT, exist_ok=True)
print(f"Active Root: {MACH3_ROOT}")

Active Root: /content/Power_BI_Spark_Labs/Power BI/Automations/Power Bi Desktop/Mach3


### 1.1 Secure Git Authentication
To push changes, we need to authenticate.
1. Go to your GitHub Settings -> Developer Settings -> Personal Access Tokens -> Tokens (classic).
2. Generate a token with `repo` permissions.
3. Add it to Colab's Secrets (left sidebar 🔑) with the name `GITHUB_TOKEN`.

In [ ]:
from google.colab import userdata
import os

# Configuration - Updated for your repository
USERNAME = "Narendra725"
REPO_NAME = "Power_BI_Spark_Labs"

try:
    token = userdata.get('GITHUB_TOKEN')
    # Re-construct the authenticated URL
    AUTH_REPO_URL = f"https://{token}@github.com/{USERNAME}/{REPO_NAME}.git"

    # Update the remote or clone if it doesn't exist
    if not os.path.exists(REPO_NAME):
        !git clone {AUTH_REPO_URL}
    else:
        %cd {REPO_NAME}
        !git remote set-url origin {AUTH_REPO_URL}
        !git pull origin main
        %cd ..
    print(f"Successfully synced: {REPO_NAME}")
except Exception as e:
    print(f"Setup Error: {e}")
    print("Please add 'GITHUB_TOKEN' to Colab Secrets (🔑) and toggle 'Notebook access'.")

## 1. Create Models from Schemas

In [115]:
import requests
import json
import re
import os
from urllib.parse import urljoin
from datamodel_code_generator import InputFileType, generate

# Define Official Fabric Schema URLs
urls = {
    "Report": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/report/3.0.0/schema.json",
    "Page": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/page/2.0.0/schema.json",
    "VisualContainer": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/visualContainer/2.3.0/schema.json",
    "Bookmark": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/bookmark/1.4.0/schema.json",
    "VersionMetadata": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/versionMetadata/1.0.0/schema.json",
    "PagesMetadata": "https://developer.microsoft.com/json-schemas/fabric/item/report/definition/pagesMetadata/1.0.0/schema.json"
}

remote_cache = {}

def resolve_type_from_ref(base_url, ref_path, current_schema):
    if ref_path.startswith('#'):
        parts = ref_path.strip('#/').split('/')
        curr = current_schema
        for p in parts: curr = curr.get(p, {})
        return curr.get('type', 'object')
    target_url = urljoin(base_url, ref_path.split('#')[0])
    if target_url not in remote_cache:
        res = requests.get(target_url); res.raise_for_status()
        remote_cache[target_url] = res.json()
    schema = remote_cache[target_url]
    if '#' in ref_path:
        parts = ref_path.split('#')[1].strip('/').split('/')
        for p in parts: schema = schema.get(p, {})
    return schema.get('type', 'object')

def clean_schema(obj, base_url, root_schema):
    if isinstance(obj, dict):
        if "description" in obj and "schema to use for an item" in obj["description"]: return {"type": "string"}
        if "$ref" in obj and isinstance(obj["$ref"], str):
            actual_type = resolve_type_from_ref(base_url, obj["$ref"], root_schema)
            return {"anyOf": [{"type": actual_type}, {"type": "null"}], "default": None}
        return {k: clean_schema(v, base_url, root_schema) for k, v in obj.items()}
    return [clean_schema(i, base_url, root_schema) for i in obj] if isinstance(obj, list) else obj

final_code = ["from __future__ import annotations", "from typing import Literal, Any, Union, List, Optional, Dict", "from pydantic import BaseModel, ConfigDict, Field, constr, RootModel"]
for name, url in urls.items():
    schema_obj = requests.get(url).json()
    cleaned = clean_schema(schema_obj, url, schema_obj)
    output = generate(json.dumps(cleaned), input_file_type=InputFileType.JsonSchema, output_model_type="pydantic_v2.BaseModel")
    block = re.sub(r'^(from __future__|from pydantic|from typing).*$', '', output, flags=re.MULTILINE)
    final_code.append(f"# --- {name} ---\n" + block.strip())

with open("/content/Power_BI_Spark_Labs/Power BI/Automations/Power Bi Desktop/Mach3/fabric_models.py", "w") as f: f.write("\n".join(final_code))
print("fabric_models.py created successfully.")

/usr/local/lib/python3.12/dist-packages/datamodel_code_generator/parser/base.py:3093: FutureWarning: The default formatters (black, isort) will be replaced by ruff in a future version. To prepare for this change, consider using: formatters=[Formatter.RUFF_FORMAT, Formatter.RUFF_CHECK]. Install ruff with: pip install 'datamodel-code-generator[ruff]'. To suppress this warning, specify formatters explicitly.
  return CodeFormatter(


fabric_models.py created successfully.


In [ ]:
import os
import json
import shutil

def save_fabric_definition(report_obj, pages_list, bookmarks_list, extra_metadata, base_output_path):
    """
    Reconstructs the full modular Fabric definition folder structure.

    :param report_obj: The Report model instance
    :param pages_list: List of (Page instance, list of VisualContainer instances)
    :param bookmarks_list: List of Bookmark instances
    :param extra_metadata: Dict containing 'pages.json' and 'bookmarks.json' raw data
    :param base_output_path: Target directory
    """
    os.makedirs(base_output_path, exist_ok=True)

    # 1. report.json
    with open(os.path.join(base_output_path, 'report.json'), 'w') as f:
        f.write(report_obj.model_dump_json(by_alias=True, exclude_none=True, indent=2))

    # 2. Pages metadata and folders
    pages_base = os.path.join(base_output_path, 'pages')
    os.makedirs(pages_base, exist_ok=True)
    if 'pages.json' in extra_metadata:
        with open(os.path.join(pages_base, 'pages.json'), 'w') as f:
            json.dump(extra_metadata['pages.json'], f, indent=2)

    for page, visuals in pages_list:
        page_folder = os.path.join(pages_base, page.name)
        os.makedirs(page_folder, exist_ok=True)
        with open(os.path.join(page_folder, 'page.json'), 'w') as f:
            f.write(page.model_dump_json(by_alias=True, exclude_none=True, indent=2))
        if visuals:
            v_base = os.path.join(page_folder, 'visuals')
            for v in visuals:
                v_data = v.root
                v_folder = os.path.join(v_base, v_data.name)
                os.makedirs(v_folder, exist_ok=True)
                with open(os.path.join(v_folder, 'visual.json'), 'w') as f:
                    f.write(v.model_dump_json(by_alias=True, exclude_none=True, indent=2))

    # 3. Bookmarks
    if bookmarks_list or 'bookmarks.json' in extra_metadata:
        bookmarks_base = os.path.join(base_output_path, 'bookmarks')
        os.makedirs(bookmarks_base, exist_ok=True)
        if 'bookmarks.json' in extra_metadata:
            with open(os.path.join(bookmarks_base, 'bookmarks.json'), 'w') as f:
                json.dump(extra_metadata['bookmarks.json'], f, indent=2)
        for b in bookmarks_list:
            # Using displayName or name for the filename; Fabric usually uses name.bookmark.json
            b_name = getattr(b, 'name', 'unknown')
            with open(os.path.join(bookmarks_base, f'{b_name}.bookmark.json'), 'w') as f:
                f.write(b.model_dump_json(by_alias=True, exclude_none=True, indent=2))

print("save_fabric_definition  to include bookmarks and metadata files.")

In [119]:
import os
import json

# Point src_path directly to the definition folder inside Mach3
src_path = os.path.join(MACH3_ROOT, 'definition')

pages_list = []
bookmarks_list = []
extra_metadata = {}

if os.path.exists(src_path):
    print(f"Loading definition from: {src_path}")
    # Load Master
    with open(os.path.join(src_path, 'report.json'), 'r') as f:
        master_report = Report(**json.load(f))

    # Load Pages
    pages_dir = os.path.join(src_path, 'pages')
    if os.path.exists(pages_dir):
        for p_folder in os.listdir(pages_dir):
            folder_path = os.path.join(pages_dir, p_folder)
            if not os.path.isdir(folder_path): continue
            page_json_path = os.path.join(folder_path, 'page.json')
            if os.path.exists(page_json_path):
                with open(page_json_path, 'r') as f: page_obj = Page(**json.load(f))
                v_list = []
                v_dir = os.path.join(folder_path, 'visuals')
                if os.path.exists(v_dir):
                    for v_f in os.listdir(v_dir):
                        v_path = os.path.join(v_dir, v_f, 'visual.json')
                        if os.path.exists(v_path):
                            with open(v_path, 'r') as f: v_list.append(VisualContainer(**json.load(f)))
                pages_list.append((page_obj, v_list))

    # Load Bookmarks
    bookmarks_dir = os.path.join(src_path, 'bookmarks')
    if os.path.exists(bookmarks_dir):
        for b_file in os.listdir(bookmarks_dir):
            if b_file.endswith('.bookmark.json'):
                with open(os.path.join(bookmarks_dir, b_file), 'r') as f:
                    bookmarks_list.append(Bookmark(**json.load(f)))

    # Instantiate Final Report object
    report = FabricReport(master_report, pages_list, bookmarks_list)
    report.get_summary()
else:
    print(f"Definition folder not found at {src_path}. Please ensure your definition folder is placed inside the Mach3 directory in your repository.")

Definition folder not found at /content/Power_BI_Spark_Labs/Power BI/Automations/Power Bi Desktop/Mach3/definition. Please ensure your definition folder is placed inside the Mach3 directory in your repository.


### Modifying a Visual and Saving the Definition
In this step, we will:
1. Find a specific visual object in our `pages_list`.
2. Update its `x` and `y` coordinates in the `position` dictionary.
3. Re-save the entire structure using `save_fabric_definition`.

In [ ]:
# 1. Target a visual on the 'Promo' page
target_page_name = 'Promo'
target_visual_name = 'fd381292653a4298f80e' # From our previous analysis

# Find the page and visual in our loaded objects
for page, visuals in pages_list:
    if page.displayName == target_page_name:
        for v_container in visuals:
            v_data = v_container.root
            if v_data.name == target_visual_name:
                print(f"Original Position for {v_data.name}: {v_data.position}")

                # 2. Modify coordinates
                v_data.position['x'] = 500.0
                v_data.position['y'] = 500.0
                print(f"Updated Position for {v_data.name}: {v_data.position}")
                break

# 3. Save to a new location
modified_dest = 'modified_definition'
save_fabric_definition(master_report, pages_list, bookmarks_list, extra_metadata, modified_dest)
print(f"\nModified definition saved to: {modified_dest}")

### Verification
Let's read the specific `visual.json` from the new directory to confirm the changes are persisted.

In [ ]:
# Identify the path to the saved visual
# Note: We need to find which folder the 'Promo' page was saved in (it uses page.name)
promo_folder = next(p.name for p, v in pages_list if p.displayName == 'Promo')
verification_path = os.path.join(modified_dest, 'pages', promo_folder, 'visuals', target_visual_name, 'visual.json')

if os.path.exists(verification_path):
    with open(verification_path, 'r') as f:
        saved_data = json.load(f)
        print(f"--- Verified Data in {verification_path} ---")
        print(f"Name: {saved_data.get('name')}")
        print(f"Position: {json.dumps(saved_data.get('position'), indent=2)}")
else:
    print("Verification file not found.")

### Walking Through the Report Structure
This code demonstrates how to iterate through the entire report hierarchy (Pages -> Visuals) and extract basic metadata using our Pydantic objects.

In [118]:
# We can now use MACH3_ROOT for all file operations
print(f"Current working directory for Fabric logic: {MACH3_ROOT}")

# Example: Updating the definition save path to use Mach3 root
# save_fabric_definition(master_report, pages_list, bookmarks_list, extra_metadata, os.path.join(MACH3_ROOT, 'definition'))

Current working directory for Fabric logic: /content/Power_BI_Spark_Labs/Power BI/Automations/Power Bi Desktop/Mach3


In [ ]:
# Redefining the report object to include bookmarks
class FabricReport:
    def __init__(self, report_metadata, pages_with_visuals, bookmarks):
        self.metadata = report_metadata
        self.pages = [FabricPage(p, v) for p, v in pages_with_visuals]
        self.bookmarks = bookmarks

    def __repr__(self):
        return f"<FabricReport: {len(self.pages)} Pages, {len(self.bookmarks)} Bookmarks>"

# Instantiate the final unified object
report = FabricReport(master_report, pages_list, bookmarks_list)

print(f"--- Report Bookmark Summary ---")
print(f"Total Bookmarks found: {len(report.bookmarks)}\n")

# Walk through the first 10 bookmarks
for i, bookmark in enumerate(report.bookmarks[:10]):
    # Accessing attributes directly from the Pydantic model
    print(f"[{i+1}] BOOKMARK: {bookmark.displayName}")
    print(f"    Internal Name: {bookmark.name}")

    # Check what this bookmark captures
    opts = bookmark.options
    captured = []
    if getattr(opts, 'targetPageVisibility', False): captured.append('Page Visibility')
    if getattr(opts, 'suppressActivePage', False) == False: captured.append('Active Page')
    if getattr(opts, 'data', False): captured.append('Data/Filters')

    print(f"    Captures: {', '.join(captured) if captured else 'None'}")

if len(report.bookmarks) > 10:
    print(f"\n... and {len(report.bookmarks) - 10} more bookmarks.")

In [99]:
from fabric_models import Report, Page, VisualContainer, Bookmark

class FabricPage:
    def __init__(self, model, visuals):
        self.model = model
        self.visuals = visuals
    def __getattr__(self, name): return getattr(self.model, name)
    def __repr__(self): return f"<FabricPage: {self.displayName} ({len(self.visuals)} visuals)>"

class FabricReport:
    def __init__(self, report_metadata, pages_with_visuals, bookmarks):
        self.metadata = report_metadata
        self.pages = [FabricPage(p, v) for p, v in pages_with_visuals]
        self.bookmarks = bookmarks
    def get_summary(self):
        print(f"--- Fabric Report Master Summary ---\nPages: {len(self.pages)} | Bookmarks: {len(self.bookmarks)}")
        for page in self.pages: print(f"- {page.displayName} ({len(page.visuals)} visuals)")
    def __repr__(self): return f"<FabricReport: {len(self.pages)} Pages>"

In [ ]:
# Executing the summary from the unified instance
report_instance.get_summary()

.

.

# UnZip the definition folder

In [111]:
import zipfile
import os

# Updated to use your repository path
repo_mach3_path = '/content/Power_BI_Spark_Labs/Power BI/Automations/Power Bi Desktop/Mach3'
zip_path = os.path.join(repo_mach3_path, 'definition.zip')
extract_path = 'fabric_report_definition'

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(f"Extracted {zip_path} to {extract_path}/")
else:
    # If the folder is already unzipped in the repo, we can point src_path directly there later
    print(f"Zip file not found at {zip_path}. Checking if definition folder exists directly...")
    if os.path.exists(os.path.join(repo_mach3_path, 'definition')):
        print("Found unzipped 'definition' folder in repository.")
    else:
        print("Could not find definition at the specified repo path.")

Zip file not found at /content/Power_BI_Spark_Labs/Power BI/Automations/Power Bi Desktop/Mach3/definition.zip. Checking if definition folder exists directly...
Could not find definition at the specified repo path.


fatal: not a git repository (or any of the parent directories): .git


In [135]:
def push_changes(commit_message="Update report definition in Mach3"):
    """Stages all changes, commits, and pushes to the repository."""
    import os

    # Navigate to the repository directory
    original_dir = os.getcwd()
    os.chdir(REPO_DIR)

    try:
        # Configure git user
        !git config --global user.email "narendradasari725@gmail.com"
        !git config --global user.name "Narendra725"

        # Add, commit and push
        !git add .
        # Check if there are changes to commit
        status = !git status --porcelain
        if status:
            !git commit -m "{commit_message}"
            !git push origin main
            print("Changes pushed successfully.")
        else:
            print("No changes to commit.")

    finally:
        # Always return to the original directory
        os.chdir(original_dir)

In [136]:
# Execute the push
push_changes("Initial setup of Mach3 structure and Fabric report definition")

[main c950db7] Initial setup of Mach3 structure and Fabric report definition
 421 files changed, 224189 insertions(+)
 create mode 100644 Power BI/Automations/Power Bi Desktop/Mach3/definition/bookmarks/1fe6d9ed1ce4e64e7b4a.bookmark.json
 create mode 100644 Power BI/Automations/Power Bi Desktop/Mach3/definition/bookmarks/231d6be9a9ce8129e9e1.bookmark.json
 create mode 100644 Power BI/Automations/Power Bi Desktop/Mach3/definition/bookmarks/3081b50dab41cd689d86.bookmark.json
 create mode 100644 Power BI/Automations/Power Bi Desktop/Mach3/definition/bookmarks/3d7b4165a4532ea34a1b.bookmark.json
 create mode 100644 Power BI/Automations/Power Bi Desktop/Mach3/definition/bookmarks/41a4dd8390d013c2b289.bookmark.json
 create mode 100644 Power BI/Automations/Power Bi Desktop/Mach3/definition/bookmarks/4cc0314d880280663807.bookmark.json
 create mode 100644 Power BI/Automations/Power Bi Desktop/Mach3/definition/bookmarks/5214773fe68b065a0dda.bookmark.json
 create mode 100644 Power BI/Automations/P